# 🛠️ MLE Data Engineering Pipeline
Welcome to the core data engineering and feature engineering pipeline. In this notebook, I demonstrate how to fetch, clean, and engineer raw financial time-series data to prepare it for machine learning volatility forecasting models.

## Step 1: Data Acquisition
I use the **Yahoo Finance API (`yfinance`)** to fetch historical daily price data. It provides high-quality Open, High, Low, Close, Volume (OHLCV) market data.

*Note: In production environments, enterprise feeds like Refinitiv or Bloomberg Terminal APIs are typically used, but `yfinance` serves as a reliable source for prototyping.*

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Fetch 2 years of daily price data for SPY
ticker = 'SPY'
raw_df = yf.download(ticker, period='2y', auto_adjust=True, progress=False)

# 2. Flatten MultiIndex columns if present (yfinance sometimes returns nested columns)
if isinstance(raw_df.columns, pd.MultiIndex):
    raw_df.columns = raw_df.columns.get_level_values(0)

# 3. Drop missing values (e.g., weekends/market holidays)
df = raw_df.dropna().copy()
df.tail()

## Step 2: Log Returns Calculation
Instead of absolute close prices, I compute **Log Returns** for training. Absolute prices are non-stationary and difficult for machine learning models to generalize.

**Rationale:** Log returns are time-additive ($R_{t_1} + R_{t_2} = R_{total}$), which is statistically crucial for building financial time-series forecasts and modeling volatility metrics.

In [ ]:
# Formula: ln(Current Close / Previous Close)
df['Return'] = np.log(df['Close'] / df['Close'].shift(1))

df[['Close', 'Return']].tail()

## Step 3: Defining the Target Variable (Y)
For supervised machine learning, we need to define the target variable. In this project, the goal is to forecast **5-day forward realized volatility**.

**Causal Alignment & Data Leakage Prevention ⚠️:**
I shift the calculated volatility backward by using `.shift(-1)`. This ensures the features at day $t$ are predicting the forward volatility from $t+1$ to $t+5$. Without this shift, the model would look ahead into future returns during training, causing data leakage.

In [ ]:
# 5-day Forward Realized Volatility (Annualised)
# sqrt(252) is used because there are 252 trading days in a year.
df['Target_Vol_5d'] = df['Return'].shift(-1).rolling(5).std() * np.sqrt(252)

df[['Return', 'Target_Vol_5d']].tail(10)

## Step 4: Feature Engineering (X)
I engineer inputs (features) that represent the underlying state and momentum of the asset:

In [ ]:
# Feature 1: Lagged Returns (captures short-term momentum memory)
df['ret_lag_1'] = df['Return'].shift(1)
df['ret_lag_5'] = df['Return'].shift(5)

# Feature 2: Historical Volatility (captures 20-day realized volatility context)
df['rv_20d'] = df['Return'].rolling(20).std() * np.sqrt(252)

# Feature 3: Relative Strength Index (RSI - measures oversold/overbought momentum)
delta = df['Close'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / (loss + 1e-8)
df['rsi_14'] = 100 - (100 / (1 + rs))

# Feature 4: Bollinger Band Width (volatility squeeze indicator)
sma20 = df['Close'].rolling(20).mean()
std20 = df['Close'].rolling(20).std()
df['bb_width'] = (2 * std20) / (sma20 + 1e-8)

df[['rv_20d', 'rsi_14', 'bb_width']].tail()

## Step 5: Final Dataset Preparation
Lagged calculations (like `.rolling(20)`) introduce missing values (`NaN`) at the beginning of the dataset, and the target shift (`.shift(-1)`) introduces `NaN` at the end.

I drop these incomplete rows to construct a clean feature matrix for training the `GradientBoostingRegressor` model.

In [ ]:
# Clean dataset ready for training
ml_dataset = df.dropna().copy()

print(f"Raw Data Rows: {len(raw_df)}")
print(f"Clean ML Rows: {len(ml_dataset)}")

ml_dataset[['Close', 'rv_20d', 'rsi_14', 'bb_width', 'Target_Vol_5d']].tail()

## Step 6: Backend ML Training Logic
The backend runs the following execution logic on the prepared dataset:
1. **Features (X)**: `['ret_lag_1', 'ret_lag_5', 'rv_20d', 'rsi_14', 'bb_width']`
2. **Target (y)**: `Target_Vol_5d`
3. **Cross-Validation**: I use `TimeSeriesSplit(n_splits=5)` to prevent look-ahead bias and evaluate temporal cross-validation metrics.
4. **Model**: Train a `GradientBoostingRegressor` to predict the forward volatility bounds.
5. **Validation**: Calculate Cross-Validation RMSE (Root Mean Squared Error) metrics displayed on the Volatility dashboard.